# Independent Hold-Out Test Evaluation

This notebook evaluates the corrected baseline CNN and VGG16 models on the official CBIS-DDSM mass test set.

The purpose of this evaluation is to obtain a stronger estimate of model generalisation using data that was not used during model training, validation, early stopping or threshold exploration.

The workflow includes:

- inspection of the official mass test metadata;
- label mapping using the same benign/malignant policy as the training data;
- duplicate and conflicting-label auditing;
- verification of patient and image independence from the existing training and validation partitions;
- baseline CNN evaluation;
- VGG16 evaluation;
- comparison of default and pre-selected candidate thresholds.

No model retraining or threshold tuning is performed using the test set.

In [1]:
import os
from pathlib import Path

DATASET_DIR = os.environ.get(
    "CBIS_DDSM_DIR",
    os.path.join("..", "data", "CBIS-DDSM")
)

dataset_root = Path(DATASET_DIR)
splits_root = Path("Splits")
models_root = Path("Models")

In [2]:
import pandas as pd

test_csv_path = dataset_root / "csv" / "mass_case_description_test_set.csv"

test_meta = pd.read_csv(test_csv_path)

print("Shape:", test_meta.shape)
print("\nColumns:")
print(test_meta.columns.tolist())

print("\nFirst 5 rows:")
display(test_meta.head())

print("\nPathology counts:")
print(test_meta["pathology"].value_counts(dropna=False))

Shape: (378, 14)

Columns:
['patient_id', 'breast_density', 'left or right breast', 'image view', 'abnormality id', 'abnormality type', 'mass shape', 'mass margins', 'assessment', 'pathology', 'subtlety', 'image file path', 'cropped image file path', 'ROI mask file path']

First 5 rows:


,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00016,4,LEFT,CC,1,mass,IRREGULAR,SPICULATED,5,MALIGNANT,5,Mass-Test_P_00016_LEFT_CC/1.3.6.1.4.1.9590.100...,Mass-Test_P_00016_LEFT_CC_1/1.3.6.1.4.1.9590.1...,Mass-Test_P_00016_LEFT_CC_1/1.3.6.1.4.1.9590.1...
1,P_00016,4,LEFT,MLO,1,mass,IRREGULAR,SPICULATED,5,MALIGNANT,5,Mass-Test_P_00016_LEFT_MLO/1.3.6.1.4.1.9590.10...,Mass-Test_P_00016_LEFT_MLO_1/1.3.6.1.4.1.9590....,Mass-Test_P_00016_LEFT_MLO_1/1.3.6.1.4.1.9590....
2,P_00017,2,LEFT,CC,1,mass,ROUND,CIRCUMSCRIBED,4,MALIGNANT,4,Mass-Test_P_00017_LEFT_CC/1.3.6.1.4.1.9590.100...,Mass-Test_P_00017_LEFT_CC_1/1.3.6.1.4.1.9590.1...,Mass-Test_P_00017_LEFT_CC_1/1.3.6.1.4.1.9590.1...
3,P_00017,2,LEFT,MLO,1,mass,ROUND,ILL_DEFINED,4,MALIGNANT,4,Mass-Test_P_00017_LEFT_MLO/1.3.6.1.4.1.9590.10...,Mass-Test_P_00017_LEFT_MLO_1/1.3.6.1.4.1.9590....,Mass-Test_P_00017_LEFT_MLO_1/1.3.6.1.4.1.9590....
4,P_00032,3,RIGHT,CC,1,mass,ROUND,OBSCURED,0,BENIGN,2,Mass-Test_P_00032_RIGHT_CC/1.3.6.1.4.1.9590.10...,Mass-Test_P_00032_RIGHT_CC_1/1.3.6.1.4.1.9590....,Mass-Test_P_00032_RIGHT_CC_1/1.3.6.1.4.1.9590....



Pathology counts:
pathology
BENIGN                     194
MALIGNANT                  147
BENIGN_WITHOUT_CALLBACK     37
Name: count, dtype: int64


## Test-set audit

The official CBIS-DDSM mass test metadata may contain multiple rows referring to the same full mammogram because individual abnormalities are recorded separately. Before model evaluation, the test metadata is therefore audited for duplicate full-mammogram references, conflicting binary labels and patient overlap with the existing training and validation partitions.

In [3]:
# Map pathology to the same binary labels used during training
label_map = {
    "BENIGN": 0,
    "BENIGN_WITHOUT_CALLBACK": 0,
    "MALIGNANT": 1
}

test_meta["label"] = test_meta["pathology"].map(label_map)

# Basic counts
print("Metadata rows:", len(test_meta))
print("Unique patients:", test_meta["patient_id"].nunique())
print("Unique full-mammogram paths:", test_meta["image file path"].nunique())

print("\nBinary label counts (metadata rows):")
print(test_meta["label"].value_counts().sort_index())

# How many mammogram paths occur more than once?
path_counts = test_meta["image file path"].value_counts()
duplicate_paths = path_counts[path_counts > 1]

print("\nFull-mammogram paths occurring more than once:")
print(len(duplicate_paths))

print("Rows belonging to repeated full-mammogram paths:")
print(test_meta["image file path"].isin(duplicate_paths.index).sum())

# Check whether any full mammogram has conflicting binary labels
conflicting_paths = (
    test_meta.groupby("image file path")["label"]
    .nunique()
)

conflicting_paths = conflicting_paths[conflicting_paths > 1]

print("\nFull mammograms with conflicting binary labels:")
print(len(conflicting_paths))

# Load the already-persisted train/validation manifests
train_df = pd.read_csv(splits_root / "train_patient_grouped.csv")
val_df = pd.read_csv(splits_root / "val_patient_grouped.csv")

train_patients = set(train_df["patient_id"].astype(str))
val_patients = set(val_df["patient_id"].astype(str))
test_patients = set(test_meta["patient_id"].astype(str))

print("\nPatient overlap checks:")
print("Test vs training patients:", len(test_patients & train_patients))
print("Test vs validation patients:", len(test_patients & val_patients))

Metadata rows: 378
Unique patients: 201
Unique full-mammogram paths: 361

Binary label counts (metadata rows):
label
0    231
1    147
Name: count, dtype: int64

Full-mammogram paths occurring more than once:
13
Rows belonging to repeated full-mammogram paths:
30

Full mammograms with conflicting binary labels:
2

Patient overlap checks:
Test vs training patients: 0
Test vs validation patients: 0


### Test-set cleaning

The same data-quality policy used for the corrected training dataset is applied to the official test set. Full mammograms associated with conflicting binary labels are excluded, and repeated metadata rows referring to the same full mammogram are collapsed to a single image-level record.

No class balancing, resampling or threshold optimisation is performed on the test set.

In [4]:
# Remove full mammograms with conflicting binary labels
clean_test_meta = test_meta[
    ~test_meta["image file path"].isin(conflicting_paths.index)
].copy()

# Deduplicate so each full mammogram appears exactly once
clean_test_meta = (
    clean_test_meta
    .drop_duplicates(subset="image file path", keep="first")
    .reset_index(drop=True)
)

print("Cleaned test mammograms:", len(clean_test_meta))
print("Cleaned test patients:", clean_test_meta["patient_id"].nunique())

print("\nCleaned binary label counts:")
print(clean_test_meta["label"].value_counts().sort_index())

print("\nRemaining duplicate full-mammogram paths:")
print(clean_test_meta["image file path"].duplicated().sum())

print("\nRemaining conflicting labels:")
remaining_conflicts = (
    clean_test_meta.groupby("image file path")["label"]
    .nunique()
)
print((remaining_conflicts > 1).sum())

Cleaned test mammograms: 359
Cleaned test patients: 200

Cleaned binary label counts:
label
0    216
1    143
Name: count, dtype: int64

Remaining duplicate full-mammogram paths:
0

Remaining conflicting labels:
0


In [5]:
dicom_info_path = dataset_root / "csv" / "dicom_info.csv"

dicom_info = pd.read_csv(dicom_info_path)

print("Shape:", dicom_info.shape)

print("\nColumns:")
print(dicom_info.columns.tolist())

print("\nFirst 5 rows:")
display(dicom_info.head())

Shape: (10237, 38)

Columns:
['file_path', 'image_path', 'AccessionNumber', 'BitsAllocated', 'BitsStored', 'BodyPartExamined', 'Columns', 'ContentDate', 'ContentTime', 'ConversionType', 'HighBit', 'InstanceNumber', 'LargestImagePixelValue', 'Laterality', 'Modality', 'PatientBirthDate', 'PatientID', 'PatientName', 'PatientOrientation', 'PatientSex', 'PhotometricInterpretation', 'PixelRepresentation', 'ReferringPhysicianName', 'Rows', 'SOPClassUID', 'SOPInstanceUID', 'SamplesPerPixel', 'SecondaryCaptureDeviceManufacturer', 'SecondaryCaptureDeviceManufacturerModelName', 'SeriesDescription', 'SeriesInstanceUID', 'SeriesNumber', 'SmallestImagePixelValue', 'SpecificCharacterSet', 'StudyDate', 'StudyID', 'StudyInstanceUID', 'StudyTime']

First 5 rows:


,file_path,image_path,AccessionNumber,BitsAllocated,BitsStored,BodyPartExamined,Columns,ContentDate,ContentTime,ConversionType,...,SecondaryCaptureDeviceManufacturerModelName,SeriesDescription,SeriesInstanceUID,SeriesNumber,SmallestImagePixelValue,SpecificCharacterSet,StudyDate,StudyID,StudyInstanceUID,StudyTime
0,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.12930...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.129308...,NaN,16,16,BREAST,351,20160426,131732.685,WSD,...,MATLAB,cropped images,1.3.6.1.4.1.9590.100.1.2.129308726812851964007...,1,23078,ISO_IR 100,20160720.0,DDSM,1.3.6.1.4.1.9590.100.1.2.271867287611061855725...,214951.0
1,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.24838...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.248386...,NaN,16,16,BREAST,3526,20160426,143829.101,WSD,...,MATLAB,full mammogram images,1.3.6.1.4.1.9590.100.1.2.248386742010678582309...,1,0,ISO_IR 100,20160720.0,DDSM,1.3.6.1.4.1.9590.100.1.2.161516517311681906612...,193426.0
2,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.26721...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.267213...,NaN,16,16,BREAST,1546,20160503,111956.298,WSD,...,MATLAB,full mammogram images,1.3.6.1.4.1.9590.100.1.2.267213171011171858918...,1,0,ISO_IR 100,20160807.0,DDSM,1.3.6.1.4.1.9590.100.1.2.291043622711253836701...,161814.0
3,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.38118...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.381187...,NaN,16,16,BREAST,97,20160503,115347.770,WSD,...,MATLAB,cropped images,1.3.6.1.4.1.9590.100.1.2.381187369611524586537...,1,32298,ISO_IR 100,20170829.0,DDSM,1.3.6.1.4.1.9590.100.1.2.335006093711888937440...,180109.0
4,CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.38118...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.381187...,NaN,8,8,Left Breast,3104,20160503,115347.770,WSD,...,MATLAB,NaN,1.3.6.1.4.1.9590.100.1.2.381187369611524586537...,1,0,ISO_IR 100,NaN,DDSM,1.3.6.1.4.1.9590.100.1.2.335006093711888937440...,NaN


In [6]:
print("Example test metadata paths:")
for path in clean_test_meta["image file path"].head(5):
    print(repr(path))

print("\nExample DICOM-info full mammogram paths:")
full_mammo_info = dicom_info[
    dicom_info["SeriesDescription"].eq("full mammogram images")
].copy()

for _, row in full_mammo_info.head(5).iterrows():
    print("file_path :", repr(row["file_path"]))
    print("image_path:", repr(row["image_path"]))
    print()

Example test metadata paths:
'Mass-Test_P_00016_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.416403281812750683720028031170500130104/1.3.6.1.4.1.9590.100.1.2.245063149211255120613007755642780114172/000000.dcm'
'Mass-Test_P_00016_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.125010935311829990124529079264106154392/1.3.6.1.4.1.9590.100.1.2.85952214611170506017891429690540035518/000000.dcm'
'Mass-Test_P_00017_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.289610447411344525237308079592285912683/1.3.6.1.4.1.9590.100.1.2.22131189612893294827907969600765582967/000000.dcm'
'Mass-Test_P_00017_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.22378064110931464632702821421529389998/1.3.6.1.4.1.9590.100.1.2.239949064412092068706566726490415129934/000000.dcm'
'Mass-Test_P_00032_RIGHT_CC/1.3.6.1.4.1.9590.100.1.2.304236983211992984124654432490357131069/1.3.6.1.4.1.9590.100.1.2.215081818713600536113960661873725083371/000000.dcm'

Example DICOM-info full mammogram paths:
file_path : 'CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.248386742010678582309005372213

In [7]:
from pathlib import PurePosixPath

# Take the first cleaned test mammogram path as an example
example_path = clean_test_meta.loc[0, "image file path"]

parts = PurePosixPath(example_path).parts

print("Original metadata path:")
print(example_path)

print("\nPath components:")
for i, part in enumerate(parts):
    print(i, part)

uid_1 = parts[-3]
uid_2 = parts[-2]

print("\nCandidate UID 1:", uid_1)
print("Candidate UID 2:", uid_2)

# Check both UIDs against relevant dicom_info columns
columns_to_check = [
    "SeriesInstanceUID",
    "StudyInstanceUID",
    "SOPInstanceUID"
]

for uid_name, uid in [("UID 1", uid_1), ("UID 2", uid_2)]:
    print(f"\nMatches for {uid_name}:")
    
    for col in columns_to_check:
        matches = dicom_info[
            dicom_info[col].astype(str) == str(uid)
        ]
        
        print(f"{col}: {len(matches)}")
        
        if len(matches) > 0:
            print(
                matches[
                    ["image_path", "SeriesDescription", col]
                ].head().to_string(index=False)
            )

Original metadata path:
Mass-Test_P_00016_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.416403281812750683720028031170500130104/1.3.6.1.4.1.9590.100.1.2.245063149211255120613007755642780114172/000000.dcm

Path components:
0 Mass-Test_P_00016_LEFT_CC
1 1.3.6.1.4.1.9590.100.1.2.416403281812750683720028031170500130104
2 1.3.6.1.4.1.9590.100.1.2.245063149211255120613007755642780114172
3 000000.dcm

Candidate UID 1: 1.3.6.1.4.1.9590.100.1.2.416403281812750683720028031170500130104
Candidate UID 2: 1.3.6.1.4.1.9590.100.1.2.245063149211255120613007755642780114172

Matches for UID 1:
SeriesInstanceUID: 0
StudyInstanceUID: 1
                                                                               image_path     SeriesDescription                                                 StudyInstanceUID
CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.245063149211255120613007755642780114172/1-271.jpg full mammogram images 1.3.6.1.4.1.9590.100.1.2.416403281812750683720028031170500130104
SOPInstanceUID: 0

Matches for UID 2

In [8]:
from pathlib import PurePosixPath, Path

# Keep only full mammogram entries from dicom_info
full_mammo_info = dicom_info[
    dicom_info["SeriesDescription"].eq("full mammogram images")
].copy()

# Build SeriesInstanceUID -> image_path mapping
series_to_jpeg = (
    full_mammo_info
    .drop_duplicates(subset="SeriesInstanceUID")
    .set_index("SeriesInstanceUID")["image_path"]
    .to_dict()
)

def extract_series_uid(metadata_path):
    parts = PurePosixPath(metadata_path).parts
    return parts[-2]

clean_test_meta["series_uid"] = clean_test_meta[
    "image file path"
].apply(extract_series_uid)

clean_test_meta["mapped_image_path"] = clean_test_meta[
    "series_uid"
].map(series_to_jpeg)

clean_test_meta["mapped_image_path"] = (
    clean_test_meta["mapped_image_path"]
    .str.replace("CBIS-DDSM/", "", regex=False)
    .str.replace("\\", "/", regex=False)
)

print("Clean test rows:", len(clean_test_meta))
print(
    "Successfully mapped:",
    clean_test_meta["mapped_image_path"].notna().sum()
)
print(
    "Unmapped:",
    clean_test_meta["mapped_image_path"].isna().sum()
)

print("\nExample mappings:")
display(
    clean_test_meta[
        [
            "patient_id",
            "pathology",
            "label",
            "image file path",
            "mapped_image_path"
        ]
    ].head()
)

Clean test rows: 359
Successfully mapped: 359
Unmapped: 0

Example mappings:


,patient_id,pathology,label,image file path,mapped_image_path
0,P_00016,MALIGNANT,1,Mass-Test_P_00016_LEFT_CC/1.3.6.1.4.1.9590.100...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.245063...
1,P_00016,MALIGNANT,1,Mass-Test_P_00016_LEFT_MLO/1.3.6.1.4.1.9590.10...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.859522...
2,P_00017,MALIGNANT,1,Mass-Test_P_00017_LEFT_CC/1.3.6.1.4.1.9590.100...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.221311...
3,P_00017,MALIGNANT,1,Mass-Test_P_00017_LEFT_MLO/1.3.6.1.4.1.9590.10...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.239949...
4,P_00032,BENIGN,0,Mass-Test_P_00032_RIGHT_CC/1.3.6.1.4.1.9590.10...,CBIS-DDSM/jpeg/1.3.6.1.4.1.9590.100.1.2.215081...


In [9]:
def local_jpeg_exists(relative_path):
    return (dataset_root / relative_path).exists()

clean_test_meta["file_exists"] = clean_test_meta[
    "mapped_image_path"
].apply(local_jpeg_exists)

print("Mapped JPEGs:", len(clean_test_meta))
print("Files found:", clean_test_meta["file_exists"].sum())
print("Files missing:", (~clean_test_meta["file_exists"]).sum())

if (~clean_test_meta["file_exists"]).any():
    print("\nMissing examples:")
    display(
        clean_test_meta.loc[
            ~clean_test_meta["file_exists"],
            ["patient_id", "mapped_image_path"]
        ].head()
    )

Mapped JPEGs: 359
Files found: 359
Files missing: 0


In [10]:
test_manifest = clean_test_meta[
    [
        "mapped_image_path",
        "label",
        "pathology",
        "patient_id"
    ]
].copy()

test_manifest = test_manifest.rename(
    columns={"mapped_image_path": "jpeg_path"}
)

splits_root.mkdir(parents=True, exist_ok=True)

test_manifest_path = splits_root / "test_independent.csv"

test_manifest.to_csv(
    test_manifest_path,
    index=False
)

print("Saved:", test_manifest_path)
print("Rows:", len(test_manifest))
print("\nLabel counts:")
print(test_manifest["label"].value_counts().sort_index())

Saved: C:\Users\Ivan\Documents\Y3S2 local\FYP\04 Implementation\Splits\test_independent.csv
Rows: 359

Label counts:
label
0    216
1    143
Name: count, dtype: int64


## Independent baseline CNN evaluation

The corrected baseline CNN is evaluated on the independent CBIS-DDSM mass test set without retraining or parameter adjustment.

Performance is assessed at:

- `0.50`, the default classification threshold;
- `0.45`, the candidate threshold previously selected from validation-set analysis.

The independent test set is not used for threshold selection.

In [11]:
import numpy as np
import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

model_path = models_root / "baseline_cnn_patient_grouped.keras"
test_manifest_path = splits_root / "test_independent.csv"

test_df = pd.read_csv(test_manifest_path)

baseline_model = tf.keras.models.load_model(model_path)

print("Test rows:", len(test_df))
print("Model loaded:", model_path.name)

print("\nTest label counts:")
print(test_df["label"].value_counts().sort_index())

Test rows: 359
Model loaded: baseline_cnn_patient_grouped.keras

Test label counts:
label
0    216
1    143
Name: count, dtype: int64


In [12]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

def load_baseline_image(jpeg_path, label):
    # Convert stored relative path to the local dataset path
    full_path = dataset_root / jpeg_path.numpy().decode("utf-8")

    image = tf.io.read_file(str(full_path))
    image = tf.image.decode_jpeg(image, channels=1)

    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    return image, label


def tf_load_baseline_image(jpeg_path, label):
    image, label = tf.py_function(
        func=load_baseline_image,
        inp=[jpeg_path, label],
        Tout=[tf.float32, tf.int64]
    )

    image.set_shape([224, 224, 1])
    label.set_shape([])

    return image, label


test_ds_baseline = tf.data.Dataset.from_tensor_slices(
    (
        test_df["jpeg_path"].astype(str).values,
        test_df["label"].astype(np.int64).values
    )
)

test_ds_baseline = (
    test_ds_baseline
    .map(tf_load_baseline_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Baseline test dataset created.")
print("Batches:", tf.data.experimental.cardinality(test_ds_baseline).numpy())

for images, labels in test_ds_baseline.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

Baseline test dataset created.
Batches: 23
Image batch shape: (16, 224, 224, 1)
Label batch shape: (16,)
Pixel range: 0.0 to 1.0


In [13]:
# Generate malignant-class probabilities
baseline_probs = baseline_model.predict(
    test_ds_baseline,
    verbose=1
).ravel()

y_true = test_df["label"].to_numpy()

print("Predictions generated:", len(baseline_probs))
print("True labels:", len(y_true))

print("\nProbability range:")
print("Min:", baseline_probs.min())
print("Max:", baseline_probs.max())
print("Mean:", baseline_probs.mean())

23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step
Predictions generated: 359
True labels: 359

Probability range:
Min: 0.014407278
Max: 0.99810106
Mean: 0.4794061


In [14]:
def evaluate_threshold(y_true, probs, threshold):
    y_pred = (probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp)
    auc = roc_auc_score(y_true, probs)

    return {
        "threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "specificity": specificity,
        "roc_auc": auc
    }


baseline_test_050 = evaluate_threshold(
    y_true,
    baseline_probs,
    0.50
)

baseline_test_045 = evaluate_threshold(
    y_true,
    baseline_probs,
    0.45
)

baseline_results = pd.DataFrame(
    [baseline_test_050, baseline_test_045]
)

display(baseline_results)

for result in [baseline_test_050, baseline_test_045]:
    print(f"\nThreshold = {result['threshold']:.2f}")
    print(
        f"TN={result['TN']}, FP={result['FP']}, "
        f"FN={result['FN']}, TP={result['TP']}"
    )
    print(f"Accuracy:    {result['accuracy']:.4f}")
    print(f"Precision:   {result['precision']:.4f}")
    print(f"Recall:      {result['recall']:.4f}")
    print(f"F1:          {result['f1']:.4f}")
    print(f"Specificity: {result['specificity']:.4f}")
    print(f"ROC-AUC:     {result['roc_auc']:.4f}")

,threshold,TN,FP,FN,TP,accuracy,precision,recall,f1,specificity,roc_auc
0,0.50,149,67,64,79,0.635097,0.541096,0.552448,0.546713,0.689815,0.645882
1,0.45,117,99,45,98,0.598886,0.497462,0.685315,0.576471,0.541667,0.645882



Threshold = 0.50
TN=149, FP=67, FN=64, TP=79
Accuracy:    0.6351
Precision:   0.5411
Recall:      0.5524
F1:          0.5467
Specificity: 0.6898
ROC-AUC:     0.6459

Threshold = 0.45
TN=117, FP=99, FN=45, TP=98
Accuracy:    0.5989
Precision:   0.4975
Recall:      0.6853
F1:          0.5765
Specificity: 0.5417
ROC-AUC:     0.6459


## Independent VGG16 evaluation

The corrected VGG16 model is evaluated on the same independent test set without retraining or parameter adjustment.

Performance is assessed at:

- `0.50`, the default classification threshold;
- `0.40`, the candidate threshold previously selected from validation-set analysis.

No threshold selection or model adjustment is performed using the independent test data.

In [15]:
vgg_model_path = models_root / "vgg16_patient_grouped.keras"

vgg_model = tf.keras.models.load_model(vgg_model_path)

print("Model loaded:", vgg_model_path.name)
print("Test rows:", len(test_df))

Model loaded: vgg16_patient_grouped.keras
Test rows: 359


In [16]:
from tensorflow.keras.applications.vgg16 import preprocess_input

def load_vgg_image(jpeg_path, label):
    full_path = dataset_root / jpeg_path.numpy().decode("utf-8")

    image = tf.io.read_file(str(full_path))
    image = tf.image.decode_jpeg(image, channels=1)

    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    # Repeat grayscale channel to create RGB-style 3-channel input
    image = tf.repeat(image, repeats=3, axis=-1)

    # Apply VGG16-specific ImageNet preprocessing
    image = preprocess_input(image)

    return image, label


def tf_load_vgg_image(jpeg_path, label):
    image, label = tf.py_function(
        func=load_vgg_image,
        inp=[jpeg_path, label],
        Tout=[tf.float32, tf.int64]
    )

    image.set_shape([224, 224, 3])
    label.set_shape([])

    return image, label


test_ds_vgg = tf.data.Dataset.from_tensor_slices(
    (
        test_df["jpeg_path"].astype(str).values,
        test_df["label"].astype(np.int64).values
    )
)

test_ds_vgg = (
    test_ds_vgg
    .map(tf_load_vgg_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("VGG16 test dataset created.")
print("Batches:", tf.data.experimental.cardinality(test_ds_vgg).numpy())

for images, labels in test_ds_vgg.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print(
        "Preprocessed pixel range:",
        float(tf.reduce_min(images)),
        "to",
        float(tf.reduce_max(images))
    )

VGG16 test dataset created.
Batches: 23
Image batch shape: (16, 224, 224, 3)
Label batch shape: (16,)
Preprocessed pixel range: -123.68000030517578 to 151.06100463867188


In [17]:
vgg_probs = vgg_model.predict(
    test_ds_vgg,
    verbose=1
).ravel()

print("Predictions generated:", len(vgg_probs))
print("True labels:", len(y_true))

print("\nProbability range:")
print("Min:", vgg_probs.min())
print("Max:", vgg_probs.max())
print("Mean:", vgg_probs.mean())

23/23 ━━━━━━━━━━━━━━━━━━━━ 15s 626ms/step
Predictions generated: 359
True labels: 359

Probability range:
Min: 0.079037495
Max: 0.98143125
Mean: 0.4740096


In [18]:
vgg_test_050 = evaluate_threshold(
    y_true,
    vgg_probs,
    0.50
)

vgg_test_040 = evaluate_threshold(
    y_true,
    vgg_probs,
    0.40
)

vgg_results = pd.DataFrame(
    [vgg_test_050, vgg_test_040]
)

display(vgg_results)

for result in [vgg_test_050, vgg_test_040]:
    print(f"\nThreshold = {result['threshold']:.2f}")
    print(
        f"TN={result['TN']}, FP={result['FP']}, "
        f"FN={result['FN']}, TP={result['TP']}"
    )
    print(f"Accuracy:    {result['accuracy']:.4f}")
    print(f"Precision:   {result['precision']:.4f}")
    print(f"Recall:      {result['recall']:.4f}")
    print(f"F1:          {result['f1']:.4f}")
    print(f"Specificity: {result['specificity']:.4f}")
    print(f"ROC-AUC:     {result['roc_auc']:.4f}")

,threshold,TN,FP,FN,TP,accuracy,precision,recall,f1,specificity,roc_auc
0,0.5,142,74,70,73,0.598886,0.496599,0.510490,0.503448,0.657407,0.626878
1,0.4,97,119,32,111,0.579387,0.482609,0.776224,0.595174,0.449074,0.626878



Threshold = 0.50
TN=142, FP=74, FN=70, TP=73
Accuracy:    0.5989
Precision:   0.4966
Recall:      0.5105
F1:          0.5034
Specificity: 0.6574
ROC-AUC:     0.6269

Threshold = 0.40
TN=97, FP=119, FN=32, TP=111
Accuracy:    0.5794
Precision:   0.4826
Recall:      0.7762
F1:          0.5952
Specificity: 0.4491
ROC-AUC:     0.6269


In [25]:
independent_test_comparison = pd.DataFrame([
    {
        "Model": "Baseline CNN",
        "Threshold": 0.50,
        "Accuracy": baseline_test_050["accuracy"],
        "Precision": baseline_test_050["precision"],
        "Recall": baseline_test_050["recall"],
        "F1": baseline_test_050["f1"],
        "Specificity": baseline_test_050["specificity"],
        "ROC-AUC": baseline_test_050["roc_auc"],
        "FN": baseline_test_050["FN"],
        "FP": baseline_test_050["FP"]
    },
    {
        "Model": "Baseline CNN",
        "Threshold": 0.45,
        "Accuracy": baseline_test_045["accuracy"],
        "Precision": baseline_test_045["precision"],
        "Recall": baseline_test_045["recall"],
        "F1": baseline_test_045["f1"],
        "Specificity": baseline_test_045["specificity"],
        "ROC-AUC": baseline_test_045["roc_auc"],
        "FN": baseline_test_045["FN"],
        "FP": baseline_test_045["FP"]
    },
    {
        "Model": "VGG16",
        "Threshold": 0.50,
        "Accuracy": vgg_test_050["accuracy"],
        "Precision": vgg_test_050["precision"],
        "Recall": vgg_test_050["recall"],
        "F1": vgg_test_050["f1"],
        "Specificity": vgg_test_050["specificity"],
        "ROC-AUC": vgg_test_050["roc_auc"],
        "FN": vgg_test_050["FN"],
        "FP": vgg_test_050["FP"]
    },
    {
        "Model": "VGG16",
        "Threshold": 0.40,
        "Accuracy": vgg_test_040["accuracy"],
        "Precision": vgg_test_040["precision"],
        "Recall": vgg_test_040["recall"],
        "F1": vgg_test_040["f1"],
        "Specificity": vgg_test_040["specificity"],
        "ROC-AUC": vgg_test_040["roc_auc"],
        "FN": vgg_test_040["FN"],
        "FP": vgg_test_040["FP"]
    }
])

display(independent_test_comparison.round({
    "Threshold": 2,
    "Accuracy": 4,
    "Precision": 4,
    "Recall": 4,
    "F1": 4,
    "Specificity": 4,
    "ROC-AUC": 4
}))

,Model,Threshold,Accuracy,Precision,Recall,F1,Specificity,ROC-AUC,FN,FP
0,Baseline CNN,0.50,0.6351,0.5411,0.5524,0.5467,0.6898,0.6459,64,67
1,Baseline CNN,0.45,0.5989,0.4975,0.6853,0.5765,0.5417,0.6459,45,99
2,VGG16,0.50,0.5989,0.4966,0.5105,0.5034,0.6574,0.6269,70,74
3,VGG16,0.40,0.5794,0.4826,0.7762,0.5952,0.4491,0.6269,32,119
